In [10]:
# ============================================================
# CREDRESOLVE — ₹10 Cr INVESTMENT ANALYSIS
# CELL 1 — LOAD VALIDATED GOLDEN DATASET
# ============================================================

import duckdb
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DB_PATH = Path("../credresolve.duckdb")
OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(exist_ok=True)

con = duckdb.connect(str(DB_PATH))

df = con.execute("""
    SELECT *
    FROM account_features
    ORDER BY account_id
""").fetchdf()

con.close()

print("=" * 80)
print("CREDRESOLVE — ₹10 Cr INVESTMENT ANALYSIS")
print("=" * 80)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique accounts:", df["account_id"].nunique())
print("Null account IDs:", df["account_id"].isna().sum())

print("\nRecovered accounts:",
      int(df["recovered_account"].sum()))

print("Unrecovered accounts:",
      int((df["recovered_account"] == 0).sum()))

print("Recovery rate:",
      round(df["recovered_account"].mean() * 100, 2), "%")

print("Recovery amount: ₹",
      round(df["total_payment_amount"].sum(), 2))

print("Portfolio outstanding: ₹",
      round(df["outstanding_amount"].sum(), 2))

display(df.head())

CREDRESOLVE — ₹10 Cr INVESTMENT ANALYSIS
Rows: 30000
Columns: 23
Unique accounts: 30000
Null account IDs: 0

Recovered accounts: 13284
Unrecovered accounts: 16716
Recovery rate: 44.28 %
Recovery amount: ₹ 1878892272.89
Portfolio outstanding: ₹ 10489035343.0


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,payment_count,total_payment_amount,recovered_account,total_calls,total_call_duration_sec,total_attempts,disposition_events,call_campaign_count,vendor_count,agent_count,targeting_events,targeting_campaigns,call_exposed,attempt_exposed,targeted_account
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,1,114222.84,1,1,44.0,5,2,1,1,1,3,2,1,1,1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,0,0.00,0,5,1238.0,2,0,5,4,5,3,3,1,1,1
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2,118181.69,1,3,1937.0,1,0,3,2,3,1,1,1,1,1
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,1,120763.00,1,0,0.0,8,1,0,0,0,1,1,0,1,1
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,0,0.00,0,0,0.0,4,0,0,0,0,0,0,0,1,0


In [11]:
# ============================================================
# CELL 2 — INVESTMENT BASELINE
# ============================================================

INVESTMENT = 100_000_000  # ₹10 Crore

portfolio_outstanding = df["outstanding_amount"].sum()
current_recovery_rate = df["recovered_account"].mean()

break_even_lift = (
    INVESTMENT / portfolio_outstanding
)

print("=" * 80)
print("₹10 Cr INVESTMENT BASELINE")
print("=" * 80)

print(f"Investment amount: ₹{INVESTMENT:,.2f}")
print(f"Portfolio outstanding: ₹{portfolio_outstanding:,.2f}")
print(f"Current recovery rate: {current_recovery_rate * 100:.2f}%")

print(
    f"Break-even recovery-rate improvement: "
    f"{break_even_lift * 100:.4f} percentage points"
)

₹10 Cr INVESTMENT BASELINE
Investment amount: ₹100,000,000.00
Portfolio outstanding: ₹10,489,035,343.00
Current recovery rate: 44.28%
Break-even recovery-rate improvement: 0.9534 percentage points


In [12]:
# ============================================================
# CELL 3 — INVESTMENT OPTIONS
# ============================================================

investment_options = pd.DataFrame({
    "investment_option": [
        "Better telephony infrastructure",
        "More collection agents",
        "AI voice automation",
        "Better borrower targeting",
        "WhatsApp / digital engagement",
        "Field operations"
    ],

    "direct_dataset_evidence": [
        "YES",
        "YES",
        "NO",
        "YES",
        "NO",
        "NO"
    ],

    "available_features": [
        "total_calls, total_call_duration_sec, total_attempts",
        "agent_count, total_calls, total_attempts",
        "No direct AI voice outcome/cost field",
        "targeting_events, targeting_campaigns, targeted_account",
        "No direct WhatsApp/digital engagement field",
        "No direct field-operation field"
    ]
})

print("=" * 80)
print("SIX INVESTMENT OPTIONS")
print("=" * 80)

display(investment_options)

SIX INVESTMENT OPTIONS


,investment_option,direct_dataset_evidence,available_features
0,Better telephony infrastructure,YES,"total_calls, total_call_duration_sec, total_at..."
1,More collection agents,YES,"agent_count, total_calls, total_attempts"
2,AI voice automation,NO,No direct AI voice outcome/cost field
3,Better borrower targeting,YES,"targeting_events, targeting_campaigns, targete..."
4,WhatsApp / digital engagement,NO,No direct WhatsApp/digital engagement field
5,Field operations,NO,No direct field-operation field


In [13]:
# ============================================================
# CELL 4 — TELEPHONY ANALYSIS
# ============================================================

df["call_exposure_band"] = pd.cut(
    df["total_calls"],
    bins=[-1, 2, 4, np.inf],
    labels=["0-2 calls", "3-4 calls", "5+ calls"]
)

call_analysis = (
    df.groupby("call_exposure_band", observed=False)
    .agg(
        accounts=("account_id", "count"),
        recovered_accounts=("recovered_account", "sum"),
        unrecovered_accounts=(
            "recovered_account",
            lambda x: (x == 0).sum()
        ),
        recovery_amount=("total_payment_amount", "sum"),
        avg_calls=("total_calls", "mean"),
        avg_attempts=("total_attempts", "mean"),
        avg_dpd=("dpd", "mean")
    )
    .reset_index()
)

call_analysis["recovery_rate_pct"] = (
    100
    * call_analysis["recovered_accounts"]
    / call_analysis["accounts"]
)

print("=" * 80)
print("TELEPHONY EXPOSURE")
print("=" * 80)

display(call_analysis)

call_analysis.to_csv(
    OUTPUT_DIR / "investment_telephony_analysis.csv",
    index=False
)

TELEPHONY EXPOSURE


,call_exposure_band,accounts,recovered_accounts,unrecovered_accounts,recovery_amount,avg_calls,avg_attempts,avg_dpd,recovery_rate_pct
0,0-2 calls,12637,5547,7090,7.889468e+08,1.398591,3.979505,56.266282,43.894912
1,3-4 calls,11815,5242,6573,7.422102e+08,3.425730,4.024630,57.104190,44.367330
2,5+ calls,5548,2495,3053,3.477353e+08,5.740988,3.994232,55.780461,44.971161


In [14]:
# ============================================================
# CELL 5 — COLLECTION AGENT ANALYSIS
# ============================================================

agent_analysis = (
    df.groupby("agent_count")
    .agg(
        accounts=("account_id", "count"),
        recovered_accounts=("recovered_account", "sum"),
        unrecovered_accounts=(
            "recovered_account",
            lambda x: (x == 0).sum()
        ),
        recovery_amount=("total_payment_amount", "sum"),
        avg_calls=("total_calls", "mean"),
        avg_attempts=("total_attempts", "mean"),
        avg_dpd=("dpd", "mean")
    )
    .reset_index()
)

agent_analysis["recovery_rate_pct"] = (
    100
    * agent_analysis["recovered_accounts"]
    / agent_analysis["accounts"]
)

print("=" * 80)
print("COLLECTION AGENT EXPOSURE")
print("=" * 80)

display(agent_analysis)

agent_analysis.to_csv(
    OUTPUT_DIR / "investment_agent_analysis.csv",
    index=False
)

COLLECTION AGENT EXPOSURE


,agent_count,accounts,recovered_accounts,unrecovered_accounts,recovery_amount,avg_calls,avg_attempts,avg_dpd,recovery_rate_pct
0,0,1695,728,967,1.047685e+08,0.064897,3.986431,55.434218,42.949853
1,1,4572,2013,2559,2.896355e+08,1.057305,3.985564,56.844269,44.028871
2,2,6799,2975,3824,4.221070e+08,2.063539,3.971319,56.417561,43.756435
3,3,6758,3007,3751,4.262145e+08,3.059041,4.046907,56.594555,44.495413
4,4,4945,2202,2743,3.084193e+08,4.064510,4.015369,56.984833,44.529828
5,5,2881,1271,1610,1.800288e+08,5.063173,4.016314,56.187435,44.116626
6,6,1419,647,772,8.864156e+07,6.070472,3.845666,56.224101,45.595490
7,7,612,284,328,3.666575e+07,7.096405,4.093137,56.344771,46.405229
8,8,224,112,112,1.569998e+07,8.062500,4.080357,56.352679,50.000000
9,9,65,32,33,4.828733e+06,9.076923,3.630769,48.076923,49.230769


In [15]:
# ============================================================
# CELL 6 — BORROWER TARGETING
# ============================================================

targeting_analysis = (
    df.groupby("targeted_account")
    .agg(
        accounts=("account_id", "count"),
        recovered_accounts=("recovered_account", "sum"),
        unrecovered_accounts=(
            "recovered_account",
            lambda x: (x == 0).sum()
        ),
        recovery_amount=("total_payment_amount", "sum"),
        avg_dpd=("dpd", "mean"),
        avg_targeting_events=("targeting_events", "mean"),
        avg_targeting_campaigns=("targeting_campaigns", "mean")
    )
    .reset_index()
)

targeting_analysis["recovery_rate_pct"] = (
    100
    * targeting_analysis["recovered_accounts"]
    / targeting_analysis["accounts"]
)

print("=" * 80)
print("BORROWER TARGETING")
print("=" * 80)

display(targeting_analysis)

targeting_analysis.to_csv(
    OUTPUT_DIR / "investment_targeting_analysis.csv",
    index=False
)

BORROWER TARGETING


,targeted_account,accounts,recovered_accounts,unrecovered_accounts,recovery_amount,avg_dpd,avg_targeting_events,avg_targeting_campaigns,recovery_rate_pct
0,0,6656,2996,3660,4.194185e+08,56.291767,0.00000,0.000000,45.012019
1,1,23344,10288,13056,1.459474e+09,56.567641,1.92769,1.915481,44.071282


In [16]:
# ============================================================
# CELL 7 — ADJUSTED TELEPHONY MODEL
# ============================================================

telephony_data = df[
    [
        "recovered_account",
        "total_calls",
        "dpd",
        "outstanding_amount",
        "principal_amount",
        "risk_segment",
        "loan_type"
    ]
].dropna().copy()

telephony_model = smf.logit(
    """
    recovered_account ~
    total_calls
    + dpd
    + outstanding_amount
    + principal_amount
    + C(risk_segment)
    + C(loan_type)
    """,
    data=telephony_data
).fit(disp=False)

print("=" * 80)
print("ADJUSTED TELEPHONY MODEL")
print("=" * 80)

print(
    "Coefficient:",
    round(
        telephony_model.params["total_calls"],
        6
    )
)

print(
    "Odds ratio:",
    round(
        np.exp(
            telephony_model.params["total_calls"]
        ),
        4
    )
)

print(
    "P-value:",
    round(
        telephony_model.pvalues["total_calls"],
        6
    )
)

ADJUSTED TELEPHONY MODEL
Coefficient: 0.012984
Odds ratio: 1.0131
P-value: 0.051541


In [17]:
# ============================================================
# CELL 8 — ADJUSTED AGENT MODEL
# ============================================================

agent_data = df[
    [
        "recovered_account",
        "agent_count",
        "dpd",
        "outstanding_amount",
        "principal_amount",
        "risk_segment",
        "loan_type"
    ]
].dropna().copy()

agent_model = smf.logit(
    """
    recovered_account ~
    agent_count
    + dpd
    + outstanding_amount
    + principal_amount
    + C(risk_segment)
    + C(loan_type)
    """,
    data=agent_data
).fit(disp=False)

print("=" * 80)
print("ADJUSTED AGENT MODEL")
print("=" * 80)

print(
    "Coefficient:",
    round(
        agent_model.params["agent_count"],
        6
    )
)

print(
    "Odds ratio:",
    round(
        np.exp(
            agent_model.params["agent_count"]
        ),
        4
    )
)

print(
    "P-value:",
    round(
        agent_model.pvalues["agent_count"],
        6
    )
)

ADJUSTED AGENT MODEL
Coefficient: 0.014662
Odds ratio: 1.0148
P-value: 0.029854


In [21]:
print(
    "Agent p-value:",
    round(
        agent_model.pvalues["agent_count"],
        6
    )
)

Agent p-value: 0.029854


In [22]:
# ============================================================
# CELL 9 — ADJUSTED TARGETING MODEL
# ============================================================

targeting_data = df[
    [
        "recovered_account",
        "targeted_account",
        "dpd",
        "outstanding_amount",
        "principal_amount",
        "risk_segment",
        "loan_type"
    ]
].dropna().copy()

targeting_model = smf.logit(
    """
    recovered_account ~
    targeted_account
    + dpd
    + outstanding_amount
    + principal_amount
    + C(risk_segment)
    + C(loan_type)
    """,
    data=targeting_data
).fit(disp=False)

print("=" * 80)
print("ADJUSTED TARGETING MODEL")
print("=" * 80)

print(
    "Targeting coefficient:",
    round(
        targeting_model.params["targeted_account"],
        6
    )
)

print(
    "Targeting odds ratio:",
    round(
        np.exp(
            targeting_model.params["targeted_account"]
        ),
        4
    )
)

print(
    "Targeting p-value:",
    round(
        targeting_model.pvalues["targeted_account"],
        6
    )
)

ADJUSTED TARGETING MODEL
Targeting coefficient: -0.038354
Targeting odds ratio: 0.9624
Targeting p-value: 0.169976


In [23]:
# ============================================================
# CELL 10 — STANDARDIZED MODEL COMPARISON
# ============================================================

# ------------------------------------------------------------
# TELEPHONY: 2 calls vs 5 calls
# ------------------------------------------------------------

calls_2 = telephony_data.copy()
calls_5 = telephony_data.copy()

calls_2["total_calls"] = 2
calls_5["total_calls"] = 5

pred_calls_2 = telephony_model.predict(calls_2)
pred_calls_5 = telephony_model.predict(calls_5)

telephony_effect_pp = (
    pred_calls_5.mean() - pred_calls_2.mean()
) * 100


# ------------------------------------------------------------
# AGENTS: 2 agents vs 3 agents
# ------------------------------------------------------------

agents_2 = agent_data.copy()
agents_3 = agent_data.copy()

agents_2["agent_count"] = 2
agents_3["agent_count"] = 3

pred_agents_2 = agent_model.predict(agents_2)
pred_agents_3 = agent_model.predict(agents_3)

agent_effect_pp = (
    pred_agents_3.mean() - pred_agents_2.mean()
) * 100


# ------------------------------------------------------------
# TARGETING: not targeted vs targeted
# ------------------------------------------------------------

targeting_no = targeting_data.copy()
targeting_yes = targeting_data.copy()

targeting_no["targeted_account"] = 0
targeting_yes["targeted_account"] = 1

pred_targeting_no = targeting_model.predict(
    targeting_no
)

pred_targeting_yes = targeting_model.predict(
    targeting_yes
)

targeting_effect_pp = (
    pred_targeting_yes.mean()
    - pred_targeting_no.mean()
) * 100


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 80)
print("STANDARDIZED MODEL COMPARISON")
print("=" * 80)

print(
    f"Telephony (2 → 5 calls): "
    f"{telephony_effect_pp:.4f} percentage points"
)

print(
    f"Agents (2 → 3 agents): "
    f"{agent_effect_pp:.4f} percentage points"
)

print(
    f"Targeting (No → Yes): "
    f"{targeting_effect_pp:.4f} percentage points"
)

print("\nIMPORTANT:")
print("These are model-based associations, not randomized causal effects.")

STANDARDIZED MODEL COMPARISON
Telephony (2 → 5 calls): 0.9615 percentage points
Agents (2 → 3 agents): 0.3614 percentage points
Targeting (No → Yes): -0.9472 percentage points

IMPORTANT:
These are model-based associations, not randomized causal effects.


In [24]:
# ============================================================
# CELL 11 — ₹10 Cr ECONOMIC SCREEN
# ============================================================

break_even_pp = (
    INVESTMENT / portfolio_outstanding
) * 100

economic_screen = pd.DataFrame({
    "investment_option": [
        "Better telephony infrastructure",
        "More collection agents",
        "Better borrower targeting"
    ],

    "model_effect_pp": [
        telephony_effect_pp,
        agent_effect_pp,
        targeting_effect_pp
    ],

    "break_even_pp": [
        break_even_pp,
        break_even_pp,
        break_even_pp
    ]
})

economic_screen["margin_vs_break_even_pp"] = (
    economic_screen["model_effect_pp"]
    - economic_screen["break_even_pp"]
)

economic_screen["meets_break_even"] = (
    economic_screen["model_effect_pp"]
    >= economic_screen["break_even_pp"]
)

print("=" * 80)
print("₹10 Cr ECONOMIC SCREEN")
print("=" * 80)

display(economic_screen)

₹10 Cr ECONOMIC SCREEN


,investment_option,model_effect_pp,break_even_pp,margin_vs_break_even_pp,meets_break_even
0,Better telephony infrastructure,0.961489,0.953377,0.008112,True
1,More collection agents,0.361396,0.953377,-0.591980,False
2,Better borrower targeting,-0.947177,0.953377,-1.900554,False


In [26]:
# ============================================================
# CELL 12 — FINAL SIX-OPTION INVESTMENT EVIDENCE
# ============================================================

final_investment_evidence = pd.DataFrame({
    "investment_option": [
        "Better telephony infrastructure",
        "More collection agents",
        "AI voice automation",
        "Better borrower targeting",
        "WhatsApp / digital engagement",
        "Field operations"
    ],

    "direct_dataset_evidence": [
        "Yes",
        "Yes",
        "No",
        "Yes",
        "No",
        "No"
    ],

    "adjusted_model_available": [
        "Yes",
        "Yes",
        "No",
        "Yes",
        "No",
        "No"
    ],

    "model_effect_pp": [
        telephony_effect_pp,
        agent_effect_pp,
        np.nan,
        targeting_effect_pp,
        np.nan,
        np.nan
    ],

    "break_even_pp": [
        break_even_pp,
        break_even_pp,
        break_even_pp,
        break_even_pp,
        break_even_pp,
        break_even_pp
    ]
})

final_investment_evidence["margin_vs_break_even_pp"] = (
    final_investment_evidence["model_effect_pp"]
    - final_investment_evidence["break_even_pp"]
)

print("=" * 80)
print("FINAL SIX-OPTION INVESTMENT EVIDENCE")
print("=" * 80)

display(final_investment_evidence)

FINAL SIX-OPTION INVESTMENT EVIDENCE


,investment_option,direct_dataset_evidence,adjusted_model_available,model_effect_pp,break_even_pp,margin_vs_break_even_pp
0,Better telephony infrastructure,Yes,Yes,0.961489,0.953377,0.008112
1,More collection agents,Yes,Yes,0.361396,0.953377,-0.591980
2,AI voice automation,No,No,NaN,0.953377,NaN
3,Better borrower targeting,Yes,Yes,-0.947177,0.953377,-1.900554
4,WhatsApp / digital engagement,No,No,NaN,0.953377,NaN
5,Field operations,No,No,NaN,0.953377,NaN


In [27]:
# ============================================================
# CELL 13 — EVIDENCE LIMITATIONS
# ============================================================

print("=" * 80)
print("EVIDENCE LIMITATIONS")
print("=" * 80)

print("""
AI VOICE AUTOMATION
-------------------
The current dataset does not contain a direct AI voice intervention,
AI voice outcome, or AI voice investment-cost variable.

Therefore, a defensible ROI cannot be calculated from this dataset.


WHATSAPP / DIGITAL ENGAGEMENT
-----------------------------
The current dataset does not contain a direct WhatsApp or digital
engagement intervention outcome or investment-cost variable.

Therefore, a defensible ROI cannot be calculated from this dataset.


FIELD OPERATIONS
----------------
The current dataset does not contain a direct field-operation
intervention outcome or investment-cost variable.

Therefore, a defensible ROI cannot be calculated from this dataset.


CAUSALITY
---------
The available analysis is observational.

Therefore, exposure-level differences and regression coefficients
should not be interpreted as proof that an intervention caused the
observed recovery outcome.


TELEPHONY
---------
Telephony is the leading directly measurable candidate because its
standardized model-based effect is 0.961489 percentage points,
slightly above the 0.953377 percentage-point break-even requirement.

However, the adjusted telephony model has p = 0.051541, which is
slightly above the conventional 5% significance threshold.

Therefore, the evidence supports a controlled pilot rather than
an unconditional ₹10 Cr rollout.
""")

EVIDENCE LIMITATIONS

AI VOICE AUTOMATION
-------------------
The current dataset does not contain a direct AI voice intervention,
AI voice outcome, or AI voice investment-cost variable.

Therefore, a defensible ROI cannot be calculated from this dataset.


WHATSAPP / DIGITAL ENGAGEMENT
-----------------------------
The current dataset does not contain a direct WhatsApp or digital
engagement intervention outcome or investment-cost variable.

Therefore, a defensible ROI cannot be calculated from this dataset.


FIELD OPERATIONS
----------------
The current dataset does not contain a direct field-operation
intervention outcome or investment-cost variable.

Therefore, a defensible ROI cannot be calculated from this dataset.


CAUSALITY
---------
The available analysis is observational.

Therefore, exposure-level differences and regression coefficients
should not be interpreted as proof that an intervention caused the
observed recovery outcome.


TELEPHONY
---------
Telephony is the leadin

In [28]:
# ============================================================
# CELL 14 — FINAL ₹10 Cr INVESTMENT DECISION
# ============================================================

print("=" * 80)
print("FINAL ₹10 Cr INVESTMENT DECISION")
print("=" * 80)

print("""
RECOMMENDATION
--------------
Do NOT commit the full ₹10 Cr immediately.

Select BETTER TELEPHONY INFRASTRUCTURE as the leading
candidate for a controlled pilot.


WHY TELEPHONY?
--------------
The model-based standardized comparison estimates:

    2 → 5 calls
    Recovery improvement = +0.961489 percentage points

The ₹10 Cr break-even requirement is:

    +0.953377 percentage points

Therefore:

    Margin above break-even = +0.008112 percentage points


IMPORTANT CAUTION
-----------------
The margin above break-even is extremely small.

The adjusted telephony model produced:

    Coefficient = 0.012984
    Odds ratio  = 1.0131
    P-value     = 0.051541

The p-value is slightly above the conventional 5% threshold.

Therefore, the analysis does NOT establish that telephony
causes the estimated recovery improvement.


OTHER DIRECTLY ANALYZED OPTIONS
-------------------------------

MORE COLLECTION AGENTS
    Model-based effect = +0.361396 percentage points
    Break-even         = +0.953377 percentage points

    This does not reach the ₹10 Cr break-even requirement.


BETTER BORROWER TARGETING
    Model-based effect = -0.947177 percentage points
    Break-even         = +0.953377 percentage points

    The adjusted targeting model is not statistically significant
    and does not support selecting targeting for the investment.


OPTIONS WITHOUT DIRECT EVIDENCE
-------------------------------

AI voice automation
WhatsApp / digital engagement
Field operations

The current dataset does not contain sufficient direct intervention
outcome and cost information to calculate defensible ROI for these
options.


FINAL DECISION
--------------

BETTER TELEPHONY INFRASTRUCTURE
        ↓
CONTROLLED PILOT
        ↓
Measure incremental recovery against a control group
        ↓
Evaluate whether the observed uplift exceeds the
0.953377 percentage-point break-even requirement
        ↓
Only then consider scaling toward the full ₹10 Cr investment.


DECISION CONFIDENCE
-------------------
LOW TO MODERATE

Reason:
Telephony is the only directly analyzed option that reaches the
calculated break-even threshold, but it exceeds break-even by only
0.008112 percentage points and its adjusted p-value is 0.051541.

A controlled experiment is therefore required before committing
the full investment.
""")

FINAL ₹10 Cr INVESTMENT DECISION

RECOMMENDATION
--------------
Do NOT commit the full ₹10 Cr immediately.

Select BETTER TELEPHONY INFRASTRUCTURE as the leading
candidate for a controlled pilot.


WHY TELEPHONY?
--------------
The model-based standardized comparison estimates:

    2 → 5 calls
    Recovery improvement = +0.961489 percentage points

The ₹10 Cr break-even requirement is:

    +0.953377 percentage points

Therefore:

    Margin above break-even = +0.008112 percentage points


IMPORTANT CAUTION
-----------------
The margin above break-even is extremely small.

The adjusted telephony model produced:

    Coefficient = 0.012984
    Odds ratio  = 1.0131
    P-value     = 0.051541

The p-value is slightly above the conventional 5% threshold.

Therefore, the analysis does NOT establish that telephony
causes the estimated recovery improvement.


OTHER DIRECTLY ANALYZED OPTIONS
-------------------------------

MORE COLLECTION AGENTS
    Model-based effect = +0.361396 percentage 

In [30]:
# ============================================================
# CELL 15 — FINAL EXPORT
# ============================================================

import duckdb
from pathlib import Path

DB_PATH = Path("../credresolve.duckdb")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

con = duckdb.connect(str(DB_PATH))

# ------------------------------------------------------------
# 1. Telephony export
# ------------------------------------------------------------

call_exposure = con.execute("""
    SELECT
        CASE
            WHEN total_calls <= 2 THEN '0-2 calls'
            WHEN total_calls <= 4 THEN '3-4 calls'
            ELSE '5+ calls'
        END AS call_exposure_band,

        COUNT(*) AS accounts,

        SUM(recovered_account) AS recovered_accounts,

        COUNT(*) - SUM(recovered_account)
            AS unrecovered_accounts,

        ROUND(SUM(total_payment_amount), 2)
            AS recovery_amount,

        ROUND(AVG(total_calls), 2)
            AS avg_calls,

        ROUND(AVG(total_attempts), 2)
            AS avg_attempts,

        ROUND(AVG(dpd), 2)
            AS avg_dpd,

        ROUND(
            100.0 * SUM(recovered_account)
            / NULLIF(COUNT(*), 0),
            2
        ) AS recovery_rate_pct

    FROM account_features

    GROUP BY
        CASE
            WHEN total_calls <= 2 THEN '0-2 calls'
            WHEN total_calls <= 4 THEN '3-4 calls'
            ELSE '5+ calls'
        END

    ORDER BY
        MIN(total_calls)
""").fetchdf()

call_exposure.to_csv(
    OUTPUT_DIR / "investment_telephony_analysis.csv",
    index=False
)


# ------------------------------------------------------------
# 2. Agent export
# ------------------------------------------------------------

agent_exposure = con.execute("""
    SELECT
        agent_count,

        COUNT(*) AS accounts,

        SUM(recovered_account)
            AS recovered_accounts,

        COUNT(*) - SUM(recovered_account)
            AS unrecovered_accounts,

        ROUND(SUM(total_payment_amount), 2)
            AS recovery_amount,

        ROUND(AVG(total_calls), 2)
            AS avg_calls,

        ROUND(AVG(total_attempts), 2)
            AS avg_attempts,

        ROUND(AVG(dpd), 2)
            AS avg_dpd,

        ROUND(AVG(outstanding_amount), 2)
            AS avg_outstanding_amount,

        ROUND(
            100.0 * SUM(recovered_account)
            / NULLIF(COUNT(*), 0),
            2
        ) AS recovery_rate_pct

    FROM account_features

    GROUP BY agent_count

    ORDER BY agent_count
""").fetchdf()

agent_exposure.to_csv(
    OUTPUT_DIR / "investment_agent_analysis.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Targeting export
# ------------------------------------------------------------

targeting_exposure = con.execute("""
    SELECT
        targeted_account,

        COUNT(*) AS accounts,

        SUM(recovered_account)
            AS recovered_accounts,

        COUNT(*) - SUM(recovered_account)
            AS unrecovered_accounts,

        ROUND(SUM(total_payment_amount), 2)
            AS recovery_amount,

        ROUND(AVG(dpd), 2)
            AS avg_dpd,

        ROUND(AVG(targeting_events), 2)
            AS avg_targeting_events,

        ROUND(AVG(targeting_campaigns), 2)
            AS avg_targeting_campaigns,

        ROUND(
            100.0 * SUM(recovered_account)
            / NULLIF(COUNT(*), 0),
            2
        ) AS recovery_rate_pct

    FROM account_features

    GROUP BY targeted_account

    ORDER BY targeted_account
""").fetchdf()

targeting_exposure.to_csv(
    OUTPUT_DIR / "investment_targeting_analysis.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Recreate economic screen from verified values
# ------------------------------------------------------------

investment_economic_screen = pd.DataFrame({
    "investment_option": [
        "Better telephony infrastructure",
        "More collection agents",
        "Better borrower targeting"
    ],

    "model_effect_pp": [
        0.961489,
        0.361396,
        -0.947177
    ],

    "break_even_pp": [
        0.953377,
        0.953377,
        0.953377
    ]
})

investment_economic_screen["margin_vs_break_even_pp"] = (
    investment_economic_screen["model_effect_pp"]
    - investment_economic_screen["break_even_pp"]
)

investment_economic_screen["meets_break_even"] = (
    investment_economic_screen["model_effect_pp"]
    >= investment_economic_screen["break_even_pp"]
)

investment_economic_screen.to_csv(
    OUTPUT_DIR / "investment_economic_screen.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Break-even summary
# ------------------------------------------------------------

investment = 100_000_000
portfolio_outstanding = con.execute("""
    SELECT SUM(outstanding_amount)
    FROM account_features
""").fetchone()[0]

current_recovery_rate = con.execute("""
    SELECT AVG(recovered_account)
    FROM account_features
""").fetchone()[0]

break_even_pp = (
    investment / portfolio_outstanding
) * 100

break_even_summary = pd.DataFrame({
    "metric": [
        "Investment amount",
        "Portfolio outstanding",
        "Current recovery rate pct",
        "Break-even uplift pp",
        "Telephony model effect pp",
        "Telephony margin vs break-even pp",
        "Agent model effect pp",
        "Targeting model effect pp"
    ],

    "value": [
        investment,
        portfolio_outstanding,
        current_recovery_rate * 100,
        break_even_pp,
        0.961489,
        0.961489 - break_even_pp,
        0.361396,
        -0.947177
    ]
})

break_even_summary.to_csv(
    OUTPUT_DIR / "investment_break_even.csv",
    index=False
)

con.close()


# ------------------------------------------------------------
# FINAL MESSAGE
# ------------------------------------------------------------

print("=" * 80)
print("INVESTMENT ANALYSIS EXPORT COMPLETE")
print("=" * 80)

print()
print("Files created successfully:")
print("1. investment_telephony_analysis.csv")
print("2. investment_agent_analysis.csv")
print("3. investment_targeting_analysis.csv")
print("4. investment_economic_screen.csv")
print("5. investment_break_even.csv")

INVESTMENT ANALYSIS EXPORT COMPLETE

Files created successfully:
1. investment_telephony_analysis.csv
2. investment_agent_analysis.csv
3. investment_targeting_analysis.csv
4. investment_economic_screen.csv
5. investment_break_even.csv


In [31]:
from pathlib import Path

sql = Path("../sql/transformations/01_account_recovery.sql").read_text()

con.execute(sql)

Task was destroyed but it is pending!
task: <Task pending name='Task-208' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\.venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-209' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\.venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\.venv\lib\site-packages\zmq\eventloop\zmqstream.py:564]>
C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\re.py:274: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  return pattern.translate(_special_chars_map)
Task was destroyed but it is pending!
task: <Task pending name='Task-209' coro=<_async_in_context.<locals>.preserv

ConnectionException: Connection Error: Connection already closed!